# Pipeline for model evaluation for the graph data extraction task

Import bibs

In [1]:
import json
import pandas as pd
from neo4j import GraphDatabase
from bert_score import score as bert_scorer
from langchain_openai import ChatOpenAI
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from dotenv import load_dotenv
import os

c:\Users\Gustavo Silveira\Documents\ITA\Management-Knowledge\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Initial Configuration

In [ ]:
# --- Load environment variable

load_dotenv()

uri = os.getenv("NEO4J_URI")
username = os.getenv("NEO4J_USERNAME")
password = os.getenv("NEO4J_PASSWORD")
openai_api_key = os.getenv("OPENAI_API_KEY")

# --- QA File path

QA_FILE_PATH = "../data/qa_neo4j.json"
RESULTS_CSV_PATH = "../data/evaluation_results_revised.csv"

# --- Choice of model to avaliate

model = "gpt-5-mini"

# --- Model who will be the judge for llm-as-a-judge
JUDGE_LLM = ChatOpenAI(model="gpt-4", temperature=0)

Defining some functions 

In [ ]:
# -- Connect to the graph

def ConnectToGraph(uri, username, password):
    try:
        graph = Neo4jGraph(
            url=uri,
            username=username,
            password=password
        )

    except Exception as e:
        print(f"Erro ao conectar-se ao Neo4j: {e}")
        graph = None
        exit()

    return graph

# --- Creation of the chain

def CreateChain(graph, llm, CYPHER_GENERATION_PROMPT, QA_PROMPT):
    """
    Cria uma cadeia de perguntas e respostas usando Cypher para o grafo Neo4j.
    """
    cypher_chain = GraphCypherQAChain.from_llm(
        llm=llm,
        graph=graph,
        cypher_prompt=CYPHER_GENERATION_PROMPT,
        qa_prompt=QA_PROMPT,
        allow_dangerous_requests=True,
        validate_cypher=True,
        return_intermediate_steps = True
    )
    return cypher_chain

# --- Creation of LLM model

def CreateLLMOpenai(model, openai_api_key):
    """
    Cria um modelo de linguagem usando o modelo de preferência.
    """
    llm = ChatOpenAI(model=model, temperature=0.2, api_key=openai_api_key) #descomentar para usar openai
    # llm = ... (hugging face) # descomentar para hugging face
    # llm = ... (other model) # descomentar para outro modelo


    CYPHER_GENERATION_TEMPLATE = """
    Você é um expert em Neo4j e Cypher. Dada uma pergunta em linguagem natural e um esquema de grafo,
    gere uma query Cypher para responder à pergunta. NÃO RESPONDA A PERGUNTA, apenas gere a query.
    Não use exemplos na query, apenas o que for necessário com base na pergunta e no esquema.
    Não coloque a query dentro de blocos de código markdown. Retorne apenas a query Cypher.
    
    Se a pergunta contiver termos que não batem exatamente com os valores armazenados no grafo,
    gere uma query Cypher que use buscas parciais ou flexíveis:
    
    - Sempre normalize comparações de texto com `toLower()`.
    - Entenda palavra militares como pessoas, não como cargo ou patente.
    - Entenda sinônimo de pessoas, por exemplo: 'Quais militares' ou 'Quais servidores/contribuidores' pode querer dizer 'Quais pessoas'.
    - Se possível, use `CONTAINS` ou `IN` para capturar sinônimos e variações.
    - Nunca use igualdade exata em strings de status ou categorias.
    - Quando comparar datas, sempre converta strings para date()/datetime().
     Exemplo: Para `após 2022`, use date(prop) > date('2022-12-31') (ou >= date('2023-01-01')).
    - Nunca compare datas como strings simples (ex.: '2022').
    - Se houver sinônimos possíveis (ex: 'conf' e 'conferência', 'artigo' e 'publicacao/publicação'), considere ambos.
    - Considere erros de grafia e acentuações.
    - Atente-se para SIGLAS (ex: USP é o mesmo que Universidade de São Paulo, ITA e Instituto Tecnológico de Aeronáutica).
    - Evite depender de igualdade exata para strings.


    Esquema do Grafo:
    {schema}

    Pergunta: {question}
    Query Cypher:
    """
    CYPHER_GENERATION_PROMPT = PromptTemplate(
        input_variables=["schema", "question"], template=CYPHER_GENERATION_TEMPLATE
    )

    # Este é o prompt para gerar a resposta final com base nos resultados da query
    QA_TEMPLATE = """Você é um assistente de IA que resume resultados de uma consulta a um banco de dados de grafo para responder a uma pergunta.
    Use os dados fornecidos no contexto para formular uma resposta clara e em linguagem natural.
    O contexto abaixo está em formato de lista de dicionários (JSON), que é o resultado da consulta.

    **Instruções Importantes:**
    1.  Se a lista no contexto estiver vazia (`[]`), responda exatamente: 'Não encontrei informações sobre isso no banco de dados.'
    2.  Caso contrário, leia as chaves e valores retornados no JSON e use-os para montar uma resposta em português claro.
    3.  Se houver múltiplos resultados, combine-os de forma natural, sem repetir estrutura JSON. 
    4.  Exemplo: se o contexto for
       `[{{"Projeto": "Sistema X", "Responsavel": "Maria Silva"}}, {{"Projeto": "Sistema Y", "Responsavel": "João Souza"}}]`
       você deve responder algo como:
       "Os projetos Sistema X (responsável: Maria Silva) e Sistema Y (responsável: João Souza) possuem documentação publicada após 2022."

    Contexto:
    {context}

    Pergunta: {question}
    Resposta útil:"""


    QA_PROMPT = PromptTemplate(input_variables=["context", "question"], template=QA_TEMPLATE)
    
    return llm, CYPHER_GENERATION_PROMPT, QA_PROMPT

Functions of assessment

In [ ]:
def can_cypher_execute(query: str, driver) -> bool:
    """Verifica se a query Cypher é executável (sintaxe correta)."""
    if not query or query.strip() == "":
        return False
    try:
        with driver.session() as session:
            session.run(f"EXPLAIN {query}")
        return True
    except Exception:
        return False
    
def evaluate_factual_consistency(question: str, gold_answer: str, generated_answer: str, judge_llm) -> str:
    """
    Usa um LLM como juiz para verificar a consistência factual.
    """
    prompt_template = """Você é um avaliador preciso e rigoroso. Sua tarefa é comparar uma 'Resposta Gerada' com uma 'Resposta de Referência (Fonte da Verdade)' no contexto de uma 'Pergunta Original'.

    Determine se a 'Resposta Gerada' é factualmente consistente com a 'Resposta de Referência'. A redação não precisa ser idêntica, mas os fatos, nomes, números e conclusões principais devem ser os mesmos.

    Responda com APENAS UMA das seguintes categorias:
    - CONSISTENTE: A resposta gerada afirma os mesmos fatos que a resposta de referência.
    - INCONSISTENTE: A resposta gerada contradiz ou afirma fatos diferentes da resposta de referência.
    - PARCIALMENTE CONSISTENTE: A resposta gerada acerta alguns fatos, mas omite outros ou contém informações adicionais não verificáveis.

    Pergunta Original:
    {question}

    Resposta de Referência (Fonte da Verdade):
    {gold_answer}

    Resposta Gerada:
    {generated_answer}

    Categoria de Consistência Factual:"""

    eval_prompt = PromptTemplate(
        input_variables=["question", "gold_answer", "generated_answer"],
        template=prompt_template
    )

    eval_chain = LLMChain(llm=judge_llm, prompt=eval_prompt)
    
    try:
        response = eval_chain.invoke({
            "question": question,
            "gold_answer": gold_answer,
            "generated_answer": generated_answer
        })
        return response['text'].strip().upper()
    except Exception as e:
        print(f"Erro na avaliação pelo LLM Juiz: {e}")
        return "ERRO_AVALIACAO"

In [ ]:
# --- PIPELINE PRINCIPAL ---

graph = ConnectToGraph(uri,username,password)
llm, cypher_prompt, qa_prompt = CreateLLMOpenai(model, openai_api_key)
chain = CreateChain(graph,llm,cypher_prompt,qa_prompt)

def run_evaluation():
    with open(QA_FILE_PATH, 'r', encoding='utf-8') as f:
        qa_dataset = json.load(f)

    neo4j_driver = GraphDatabase.driver(uri, auth=(username, password))
    evaluation_results = []
    total_items = len(qa_dataset)

    print("Iniciando o pipeline de avaliação revisado...")

    for i, item in enumerate(qa_dataset):
        print(f"Processando questão {i+1}/{total_items}: \"{item['question']}\"")

        question = item['question']
        gold_cypher = item['gold_cypher']
        gold_answer = item['gold_answer']

        try:
            llm_response = chain.invoke({"query": question})
            generated_cypher = llm_response.get('intermediate_steps', [{}])[0].get('query', 'N/A')
            generated_answer = llm_response.get('result', 'N/A')

            # 1. Avaliar a query Cypher (apenas se executa)
            cypher_executable = can_cypher_execute(generated_cypher, neo4j_driver)

            # 2. Avaliar a resposta textual
            P, R, F1 = bert_scorer([generated_answer], [gold_answer], lang="pt", model_type='bert-base-multilingual-cased')
            bert_f1 = F1.item()

            factual_consistency = evaluate_factual_consistency(question, gold_answer, generated_answer, JUDGE_LLM)

            evaluation_results.append({
                "id": item['id'],
                "question": question,
                "gold_cypher": gold_cypher,
                "generated_cypher": generated_cypher,
                "cypher_executable": cypher_executable,
                "gold_answer": gold_answer,
                "generated_answer": generated_answer,
                "bert_score_f1": bert_f1,
                "factual_consistency": factual_consistency
            })
        
        except Exception as e:
            print(f"!!! ERRO GERAL ao processar a questão ID {item['id']}: {e}")
            evaluation_results.append({
                "id": item['id'], "question": question, "generated_cypher": "ERRO",
                "cypher_executable": False, "generated_answer": f"ERRO NA EXECUÇÃO: {e}",
                "bert_score_f1": 0.0, "factual_consistency": "ERRO_EXECUCAO"
            })

    neo4j_driver.close()
    df = pd.DataFrame(evaluation_results)
    df.to_csv(RESULTS_CSV_PATH, index=False, encoding='utf-8-sig')
    print(f"\nResultados detalhados salvos em: {RESULTS_CSV_PATH}")

    # --- ANÁLISE DOS RESULTADOS ---
    cypher_exec_rate = df['cypher_executable'].mean() * 100
    average_bert_score = df['bert_score_f1'].mean()
    consistency_distribution = df['factual_consistency'].value_counts(normalize=True) * 100
    
    print("\n--- RESUMO DA AVALIAÇÃO REVISADA ---")
    print(f"Taxa de Execução das Queries Cypher: {cypher_exec_rate:.2f}%")
    print(f"BERTScore F1 Médio (Similaridade Semântica): {average_bert_score:.4f}")
    print("\nDistribuição da Consistência Factual:")
    print(consistency_distribution.to_string())
    print("-------------------------------------\n")
    
    print("Amostra de Respostas marcadas como INCONSISTENTES:")
    inconsistent_df = df[df['factual_consistency'] == 'INCONSISTENTE']
    print(inconsistent_df[['id', 'question', 'gold_answer', 'generated_answer']].head())